In [ ]:
!pip install -q transformers==4.57.6

In [1]:
from collections import defaultdict
import re
from string import punctuation

import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
from torch.nn import CosineSimilarity
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForMaskedLM, AutoTokenizer
from tqdm.notebook import tqdm

In [7]:
model = SentenceTransformer("sachinn1/xl-durel", processor_kwargs={"truncation": False}, device="cuda")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
c1 = pd.read_csv("drive/MyDrive/generated_definitions/english/dpo-clustered/english-corpus1.tsv.gz", sep="\t", compression="gzip")
c1.head()

,0,1,2
0,attack,"tiring breeze , and the grief - like and hollo...","An attempt to harm, damage, debilitate, or des..."
1,attack,"Ik saai , Ik hab mien dout , "" and having thus...","A violent assault, whether physical or verbal."
2,attack,"Having determined instantly , in his own mind ...","A violent engagement of bodies, either by land..."
3,attack,What is remarkable in these and in all the oth...,"A violent assault, for example on a building; ..."
4,attack,Just as they had taken their seats around the ...,An attempt to detract from the worth or credib...


In [5]:
c2 = pd.read_csv("drive/MyDrive/generated_definitions/english/dpo-clustered/english-corpus2.tsv.gz", sep="\t", compression="gzip")
c2.head()

,0,1,2
0,attack,"to listen to words of peace , and avert by ful...",An attempt to detract from the worth or credib...
1,attack,"The tabloid TV shows , like "" Hard Copy "" and ...","An attempt to cause damage, injury to, or deat..."
2,attack,We believe that the United States should pursu...,An attempt to detract from the worth or credib...
3,attack,"However , when one discovers that Snyder 's ga...",An attempt to detract from the worth or credib...
4,attack,Yet the reality today is worse than fiction : ...,"(Medicine) An acute myocardial infarction, som..."


In [6]:
class TargetDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):

        return self.examples[i]

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [32]:
def create_usage_matrix(examples_list):

    dataset = TargetDataset(examples_list)
    loader = DataLoader(dataset, batch_size=64, num_workers=0)
    usage_matrix = np.empty((1, 1024))
    for batch in loader:

        embedding = model.encode(batch)
        usage_matrix = np.concatenate((usage_matrix, embedding), axis=0)
    usage_matrix = usage_matrix[1:, :]
    assert usage_matrix.shape[0] > 0
    return usage_matrix

In [10]:
def process(s, target):
    start_idx = s.find(target)
    result = s[:start_idx] + "<t>" + target + " </t>" + s[start_idx + len(target):]
    return result

In [18]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [27]:
X = [[0, 0, 0], [1, 1, 1]]
Y = [[1, 0, 0], [1, 1, 0]]
1 - float(np.mean(cosine_similarity(X, Y)))

0.651538287470662

In [33]:
cos = CosineSimilarity(dim=1, eps=1e-6)
model.eval()
table_2 = defaultdict(list)
for target, t_result in tqdm(c1.groupby("0")):
    year_predictions = []
    y_result = c2[c2["0"] == target]
    t_result["proc"] = t_result["1"].apply(lambda x: process(x, target))
    y_result["proc"] = y_result["1"].apply(lambda x: process(x, target))
    usage_matrix1 = create_usage_matrix(t_result["proc"].to_list())
    usage_matrix2 = create_usage_matrix(y_result["proc"].to_list())
    samples_1 = usage_matrix1.shape[0]
    samples_2 = usage_matrix2.shape[0]
    min_samples = min(samples_1, samples_2)
    usage_matrix1 = usage_matrix1[:min_samples, :]
    usage_matrix2 = usage_matrix2[:min_samples, :]


    sim = np.mean(cosine_similarity(usage_matrix1, usage_matrix2))
    table_2["target"].append(target)
    apd = 1 - float(sim)
    table_2["apd"].append(apd)

pd.DataFrame(table_2).to_csv("drive/MyDrive/apd-semeval.tsv", sep="\t", index=False)

  0%|          | 0/37 [00:00<?, ?it/s]

/tmp/ipykernel_19413/3493419780.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_result["proc"] = y_result["1"].apply(lambda x: process(x, target))
/tmp/ipykernel_19413/3493419780.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y_result["proc"] = y_result["1"].apply(lambda x: process(x, target))
/tmp/ipykernel_19413/3493419780.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 